In [150]:
import numpy as np
import psutil
import subprocess
import random as rd
import pickle
import gc
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import LogarithmicMapper

def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")


def single_line (lines):
    return ''.join(lines.splitlines())

In [151]:
n_x = 4
n_y = 1
n_site = n_x * n_y
n_qubit = n_site
dim = 2**n_qubit

n_qubit_circuit = n_qubit + 1

n_dimer = n_site//2

core  = 0
cores = 1

In [152]:
seed_transpiler = 42

In [153]:
from qiskit import QuantumRegister, QuantumCircuit
qr_circuit = QuantumRegister(n_qubit_circuit,name='q')
#qr = qr_circuit[1:]
#qm = qr_circuit[0]
#indx_qm = 0
indx_qm = n_qubit//2
qm = qr_circuit[indx_qm]
qr = qr_circuit[:indx_qm] + qr_circuit[indx_qm+1:]
#print(qm,qr)

In [154]:
J                   = 1.0
Delta               = -1.0
h                   = 0.0

In [155]:
if (n_site<10):
    t_inters = [0.0, 1.0]
else:
    t_inters            = [0.0, 0.75, 1.0]
#t_inters            = [0.0, 1.0]
n_inter             = len(t_inters)
hamiltonians        = []
mapper = LogarithmicMapper()
for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    hamiltonian_list = []
    # intra-dimer terms
    for i in range(0,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*Delta/4)
        hamiltonian_list.append(term)
        term = ('Z',[ii],-h/2)
        hamiltonian_list.append(term)
        term = ('Z',[jj],-h/2)
        hamiltonian_list.append(term)
    # inter-dimer terms
    for i in range(1,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*t_inter*Delta/4)
        hamiltonian_list.append(term)
    print(hamiltonian_list)
    hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonians.append(hamiltonian.simplify())

    if (core==0):
        print(t_inter, single_line(str(hamiltonians[i_inter])))
        print('')

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [1, 2], -0.0), ('YY', [1, 2], -0.0), ('ZZ', [1, 2], 0.0)]
0.0 SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'XXII', 'YYII', 'ZZII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [1, 2], -0.25), ('YY', [1, 2], -0.25), ('ZZ', [1, 2], 0.25)]
1.0 SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'XXII', 'YYII', 'ZZII', 'IXXI', 'IYYI', 'IZZI'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])



In [156]:
init_circuit = QuantumCircuit(qr_circuit)
# init circuit for staring from dimers
n_dimer_half = n_dimer//2
index_attached_to_qm = n_qubit//2-1

#  geometry ; qm -- qr[n_qubit//2-1] -- ... -- qr[1] -- qr[0]
#                   \--qr[n_qubit//2] -- ... -- qr[n_qubit-1]
# apply cxs first
init_circuit.cx(qm,qr[index_attached_to_qm])
for i_qubit in range(index_attached_to_qm,1,-1):
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
for i_qubit in range(index_attached_to_qm,n_qubit-2):
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])


# apply chs + z + cx
init_circuit.ch(qr[1],qr[0])
init_circuit.cx(qr[0],qr[1])
for i_dimer in range(1,n_dimer-1):
    i_qubit = 2*i_dimer
    init_circuit.ch(qr[i_qubit+1],qr[i_qubit])
    init_circuit.z(qr[i_qubit])
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])

init_circuit.ch(qr[-2],qr[-1])
init_circuit.cx(qr[-1],qr[-2])

#
#for i_dimer in range(1,n_dimer_half+1):
#    i_qubit = 2*(i_dimer+n_dimer_half)
#    init_circuit.s(qr[i_qubit+1])
#    init_circuit.h(qr[i_qubit+1])
#    init_circuit.t(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])
#    init_circuit.tdg(qr[i_qubit+1])
#    if (i_dimer<n_dimer_half):
#        init_circuit.sx(qr[i_qubit+1])
#        init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])
#        init_circuit.x(qr[i_qubit+1])
#        init_circuit.h(qr[i_qubit+1])
#    else:
#        init_circuit.h(qr[i_qubit+1])
#        init_circuit.sdg(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit+1],qr[i_qubit])




print(init_circuit)
init_circuit_inv = init_circuit.inverse()
print(init_circuit_inv)

               ┌───┐     
q_0: ──────────┤ H ├──■──
     ┌───┐     └─┬─┘┌─┴─┐
q_1: ┤ X ├──■────■──┤ X ├
     └─┬─┘  │       └───┘
q_2: ──■────┼────────────
          ┌─┴─┐     ┌───┐
q_3: ─────┤ X ├──■──┤ X ├
          └───┘┌─┴─┐└─┬─┘
q_4: ──────────┤ H ├──■──
               └───┘     
          ┌───┐          
q_0: ──■──┤ H ├──────────
     ┌─┴─┐└─┬─┘     ┌───┐
q_1: ┤ X ├──■────■──┤ X ├
     └───┘       │  └─┬─┘
q_2: ────────────┼────■──
     ┌───┐     ┌─┴─┐     
q_3: ┤ X ├──■──┤ X ├─────
     └─┬─┘┌─┴─┐└───┘     
q_4: ──■──┤ H ├──────────
          └───┘          


In [157]:
n_hamiltonians = len(hamiltonians)
if (core==0):
    print('# Hamiltonian differences')
hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs[alpha])))

if (core==0):
    print('# Hamiltonian differences_list')
hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_list[alpha])

if (core==0):
    print('# Reduced Hamiltonian differences_list')
hamiltonian_diffs_reduced = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_list = []
    if (n_site>=4) and (n_dimer%2==0):
        # for 4 site
        # pos # 1
        factor_XX = 2  # 1 for XX, 1 for YY
        factor_ZZ = -1 # -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm
        jj = index_attached_to_qm+1
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=6) and (n_dimer%2==1):
        # for 6 site
        # pos # 1
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm
        jj = index_attached_to_qm-1
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=8) and (n_dimer%2==0):
        # for 8 site
        # pos # 2
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm-1
        jj = index_attached_to_qm-2
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)
    if (n_site>=10) and (n_dimer%2==1):
        # for 10 site
        # pos # 2
        factor_XX = 2*2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
        factor_ZZ = -2 # 2 for symmetric two bonds, -sign from XX <> -ZZ symmetry for Delta=-1
        ii = index_attached_to_qm-2
        jj = index_attached_to_qm-3
        term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*(factor_XX+Delta*factor_ZZ))
        hamiltonian_list.append(term)

    print(hamiltonian_list)
    dH = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonian_diffs_reduced.append(dH.simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs_reduced[alpha])))
if (core==0):
    print('# Hamiltonian differences_reduced_list')
hamiltonian_diffs_reduced_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_reduced_list.append(hamiltonian_diffs_reduced[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_reduced_list[alpha])

# Hamiltonian differences
0 SparsePauliOp(['IXXI', 'IYYI', 'IZZI'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j])
# Hamiltonian differences_list
[('IXXI', (-0.25+0j)), ('IYYI', (-0.25+0j)), ('IZZI', (0.25+0j))]
# Reduced Hamiltonian differences_list
[('XX', [1, 2], -0.75)]
0 SparsePauliOp(['IXXI'],              coeffs=[-0.75+0.j])
# Hamiltonian differences_reduced_list
[('IXXI', (-0.75+0j))]


In [158]:
nmc = 300
beta = 2.0
n_shot = 1024

n_obs = 3
# 0; norm, 1; dE1, 2; dE2
O_timelists         = [[[[] for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]

#exact_dir = '../exact'
#with open(exact_dir+'/XXZ10.time.binary','rb') as file_:
#    [O_timelists] = pickle.load(file_)

exact_dir = '../../'+str(n_site)+'/exact'
with open(exact_dir+'/XXZ'+str(n_site)+'.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

In [159]:
observable = SparsePauliOp.from_sparse_list([("Z", [indx_qm], 1)], num_qubits=n_qubit_circuit)
print(observable)

SparsePauliOp(['IIZII'],
              coeffs=[1.+0.j])


In [160]:
def Apply_R_XXplusYYplusZZ(theta_x,theta_y,theta_z, jj, qc_: QuantumCircuit):
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(theta_z,qr[jj])
    qc_.rz(theta_x+np.pi/2,qr[jj+1])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(-theta_y,qr[jj])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.sx(qr[jj])
    qc_.sxdg(qr[jj+1])
    


In [161]:
def ApplyConsecutiveTrotterEvolution_from_dimer_solution(times, alphas, n_trotters, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    # only consider indx==1 case
    # intra-dimer evolution first
    # skip the first evolution
    # phase corresponding to this part should be considered properly.
    # for i_qubit in range(0,n_qubit,2):
    #     Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
    #                            thetas_yy_intra[0]/2.0,
    #                            thetas_zz_intra[0]/2.0,
    #                            i_qubit, qc_)
    for i_time in range(n_time-1):
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                   thetas_yy_inter[i_time],
                                   thetas_zz_inter[i_time],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[i_time]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                   (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                   (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                   i_qubit, qc_)
    # last evolution
    # inter-dimer evolution
    for i_qubit in range(1,n_qubit-1,2):
        Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                               thetas_yy_inter[-1],
                               thetas_zz_inter[-1],
                               i_qubit, qc_)
    for i_trotter in range(n_trotters[-1]-1):
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
    # intra-dimer evolution
    for i_qubit in range(0,n_qubit,2):
        Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
                               (thetas_yy_intra[-1])/2.0,
                               (thetas_zz_intra[-1])/2.0,
                               i_qubit, qc_)

def ApplyConsecutiveTrotterEvolution_to_dimer_solution(times, alphas, n_trotters, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    # only consider indx==1 case
    # intra-dimer evolution first
    for i_qubit in range(0,n_qubit,2):
        Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
                               thetas_yy_intra[0]/2.0,
                               thetas_zz_intra[0]/2.0,
                               i_qubit, qc_)
    for i_time in range(n_time-1):
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                   thetas_yy_inter[i_time],
                                   thetas_zz_inter[i_time],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[i_time]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                   (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                   (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                   i_qubit, qc_)
    # last evolution
    # inter-dimer evolution
    for i_qubit in range(1,n_qubit-1,2):
        Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                               thetas_yy_inter[-1],
                               thetas_zz_inter[-1],
                               i_qubit, qc_)
    for i_trotter in range(n_trotters[-1]-1):
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
    # intra-dimer evolution
    # skip the last evolution
    # phase corresponding to this part should be considered properly.
    # for i_qubit in range(0,n_qubit,2):
    #     Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
    #                            (thetas_yy_intra[-1])/2.0,
    #                            (thetas_zz_intra[-1])/2.0,
    #                            i_qubit, qc_)

In [162]:
def ApplyConsecutiveTrotterEvolution(times, alphas, n_trotters, indx, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    if (indx==0):
        # inter-dimer evolution first
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[0]/2.0,
                                   thetas_yy_inter[0]/2.0,
                                   thetas_zz_inter[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_inter[i_time]+thetas_xx_inter[i_time+1])/2.0,
                                       (thetas_yy_inter[i_time]+thetas_yy_inter[i_time+1])/2.0,
                                       (thetas_zz_inter[i_time]+thetas_zz_inter[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_inter[-1])/2.0,
                                   (thetas_yy_inter[-1])/2.0,
                                   (thetas_zz_inter[-1])/2.0,
                                   i_qubit, qc_)
    elif (indx==1):
        # intra-dimer evolution first
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
                                   thetas_yy_intra[0]/2.0,
                                   thetas_zz_intra[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                       (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                       (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
                                   (thetas_yy_intra[-1])/2.0,
                                   (thetas_zz_intra[-1])/2.0,
                                   i_qubit, qc_)

In [163]:
# read exact values
nsec = n_qubit//2 + 1
exact_dir = '../../'+str(n_site)+'/exact'
norms_exact = np.zeros((nsec,n_hamiltonians), dtype=float)
eigen_energies_exact = np.zeros((nsec,n_hamiltonians),dtype=float)
with open(exact_dir + '/exact.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        for isec in range(nsec):
            norms_exact[isec,alpha] = float(ls[isec+1])
            eigen_energies_exact[isec,alpha] = float(ls[isec+nsec+1])
        alpha += 1
for isec in range(nsec):
    for alpha in range(n_hamiltonians):
        print(isec,alpha,norms_exact[isec,alpha],eigen_energies_exact[isec,alpha])

0 0 1.0 0.5
0 1 1.0 0.75
1 0 1.0 -0.5
1 1 0.4267766952966368 -0.9571067811865475
2 0 1.0 -1.5
2 1 0.9330127018922203 -1.6160254037844386


In [164]:
def NumberOfTrotterSteps(alpha):
    return 2
if (core==0):
    print('# of Trotter Steps for each alpha')
    for alpha_ in range(1,n_hamiltonians):
        print(NumberOfTrotterSteps(alpha_))

# of Trotter Steps for each alpha
2


In [165]:
#from qiskit_ibm_runtime import QiskitRuntimeService
# 
## Load saved credentials
#service = QiskitRuntimeService()
#backend_name = "ibm_torino" 
#backend = service.backend(name=backend_name)

# use fake backend
# from qiskit_ibm_runtime.fake_provider import FakeTorino
# backend = FakeTorino()
# from qiskit_ibm_runtime import EstimatorOptions
# from qiskit_ibm_runtime import EstimatorV2 as Estimator
# options = EstimatorOptions()
# options.default_shots = n_shot
# options.resilience_level = 1
# options.dynamical_decoupling.enable = True
# options.dynamical_decoupling.sequence_type = "XY4"
# estimator = Estimator(backend, options=options)
# max_circuits = 300 # backend.max_circuits


In [166]:
#initial_layout = [95,96,97,110,98,99,92] # seems to best at 241111, 08:12 PM

In [167]:
#from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
#pm3 = generate_preset_pass_manager(initial_layout=initial_layout, backend=backend, optimization_level=3, seed_transpiler=seed_transpiler)
#pm1 = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=seed_transpiler)

In [168]:
from qiskit import qpy
backend_options = {'method':'statevector', 'max_parallel_threads':1}
if (core==0):
    code_run_pass_manager = '''from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import qpy
from qiskit_aer import AerSimulator
backend_options = '''+ str(backend_options) +'''
sim = AerSimulator(**backend_options)
pass_manager = generate_preset_pass_manager(2, sim)
with open('circuits.qpy', 'rb') as file_:
    circuits = qpy.load(file_)
    n_circuit = len(circuits)
    circuits_opt = []
    for i_circuit in range(n_circuit):
        circuit_opt = pass_manager.run(circuits[i_circuit])
        circuits_opt.append(circuit_opt)
with open ('circuits_opt.qpy', 'wb') as file_:
    qpy.dump(circuits_opt,file_)
    '''
    
    with open('run_pass_manager.py', 'w') as file_:
        file_.write(code_run_pass_manager)

In [169]:
# make a circuit
nhd = len(hamiltonian_diffs_reduced_list[0]) # same difference elements for all
circuits = []
# for n_qubit>=4, even dimer
# CXX, at pos # 1
if (n_site>=4) and (n_dimer%2==0):
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
    circuits.append(circuit)
# for n_qubit>=6, odd dimer
# CXX, at pos # 1
if (n_site>=6) and (n_dimer%2==1):
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuit.cx(qm,qr[index_attached_to_qm])
    circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
    circuits.append(circuit)
# for n_qubit>=8, even dimer
# CXX, at pos # 2
if (n_site>=8) and (n_dimer%2==0):
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuit.cx(qm,qr[index_attached_to_qm-1])
    circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
    circuits.append(circuit)
if (n_site>=10) and (n_dimer%2==1):
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
    circuit.cx(qm,qr[index_attached_to_qm-2])
    circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
    circuits.append(circuit)
# for n_qubit>=12, even dimer
# CXX, at pos # 2
if (n_site>=12) and (n_dimer%2==0):
    circuit = QuantumCircuit(qr_circuit)
    circuit.cx(qr[index_attached_to_qm-3],qr[index_attached_to_qm-4])
    circuit.cx(qm,qr[index_attached_to_qm-3])
    circuit.cx(qr[index_attached_to_qm-3],qr[index_attached_to_qm-4])
    circuits.append(circuit)


# # CZZ, at pos # 1
# circuit = QuantumCircuit(qr_circuit)
# circuit.h(qr[index_attached_to_qm])
# circuit.h(qr[index_attached_to_qm+1])
# circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
# circuit.cx(qm,qr[index_attached_to_qm])
# circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm+1])
# circuit.h(qr[index_attached_to_qm+1])
# circuit.h(qr[index_attached_to_qm])
# circuits.append(circuit)

# CZZ, at pos # 1
#circuit = QuantumCircuit(qr_circuit)
#circuit.h(qr[index_attached_to_qm])
#circuit.h(qr[index_attached_to_qm-1])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.h(qr[index_attached_to_qm-1])
#circuit.h(qr[index_attached_to_qm])
#circuits.append(circuit)

# for n_qubit>=8, odd dimer
# # CXX, at pos # 2
#circuit = QuantumCircuit(qr_circuit)
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuits.append(circuit)
## CZZ, at pos # 2
#circuit = QuantumCircuit(qr_circuit)
#circuit.h(qr[index_attached_to_qm-2])
#circuit.h(qr[index_attached_to_qm-3])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm-1],qr[index_attached_to_qm-2])
#circuit.cx(qr[index_attached_to_qm-2],qr[index_attached_to_qm-3])
#circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
#circuit.cx(qm,qr[index_attached_to_qm])
#circuit.h(qr[index_attached_to_qm-2])
#circuit.h(qr[index_attached_to_qm-3])
#circuits.append(circuit)

In [170]:
for circuit in circuits:
    print(circuit)

                    
q_0: ───────────────
          ┌───┐     
q_1: ──■──┤ X ├──■──
       │  └─┬─┘  │  
q_2: ──┼────■────┼──
     ┌─┴─┐     ┌─┴─┐
q_3: ┤ X ├─────┤ X ├
     └───┘     └───┘
q_4: ───────────────
                    


In [171]:
if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    pass_manager_run = subprocess.run(['python3', 'run_pass_manager.py'])

In [172]:
with open('circuits_opt.qpy', 'rb') as file_:
    paulis_opt = qpy.load(file_)
for circuit in paulis_opt:
    print(circuit)

                    
q_0: ───────────────
          ┌───┐     
q_1: ──■──┤ X ├──■──
       │  └─┬─┘  │  
q_2: ──┼────■────┼──
     ┌─┴─┐     ┌─┴─┐
q_3: ┤ X ├─────┤ X ├
     └───┘     └───┘
q_4: ───────────────
                    


In [173]:
#paulis_opt = []
#for circuit in circuits:
#    circuit_opt = pm3.run(circuit)
#    paulis_opt.append(circuit_opt) 

In [174]:
#paulis_opt[0].draw(idle_wires=False)

In [175]:
#paulis_opt[1].draw(idle_wires=False)

In [176]:
# find the position of CZ[qm,qr]
# for pauli0, it is 7
# for pauli1, it is 6
#indx_cz = [0 for _ in range(nhd)]
#indx_cz[0] = 7
#indx_cz[1] = 6
## check
#k = 0
#for inst in paulis_opt[0].data:
#    if (k==indx_cz[0]):
#        print(inst)
#    k += 1 
#k = 0
#for inst in paulis_opt[1].data:
#    if (k==indx_cz[1]):
#        print(inst)
#    k += 1 

In [177]:
# get a duration
# from qiskit.transpiler import InstructionDurations
# dur = InstructionDurations.from_backend(backend)
# #print(dur)
# t_cx_m1 = dur.get('cz',(initial_layout[indx_qm],initial_layout[index_attached_to_qm]))
# t_cx_m2 = dur.get('cz',(initial_layout[index_attached_to_qm-2],initial_layout[index_attached_to_qm-3]))
# print(t_cx_m1*dur.dt)
# print(t_cx_m2*dur.dt)

In [178]:
#observable_device = observable.apply_layout(paulis_opt[0].layout)
#n_qubit_device = backend.num_qubits
#pauli = observable_device.paulis[0]
#for k in range(n_qubit_device):
#    p = pauli[k]
#    if (str(p)=='Z'):
#        print(k,p)

In [179]:
#def ApplyControlledPauliGate(ihd, qc_):
#    for inst in paulis_opt[ihd].data:
#        qc_.append(inst.operation, inst.qubits, inst.clbits)
#empty_paulis_opt = []
#for ihd in range(nhd):
#    circuit = QuantumCircuit(qr_circuit)
#    circuit = pm3.run(circuit)
#    ii = 0
#    for inst in paulis_opt[ihd].data:
#        #print(inst)
#        if ii==indx_cz[ihd]:
#            print(inst)
#            circuit.barrier()
#            circuit.delay(t_cx_m1,initial_layout[2])
#            circuit.delay(t_cx_m1,initial_layout[1])
#            circuit.barrier()
#        else:
#            circuit.append(inst.operation, inst.qubits, inst.clbits)
#        ii += 1
#    empty_paulis_opt.append(circuit)
#def ApplyEmptyPauliGate(ihd, qc_):
#    for inst in empty_paulis_opt[ihd].data:
#        qc_.append(inst.operation, inst.qubits, inst.clbits)

In [180]:
def ApplyControlledPauliGate(ihd, qc_):
    for inst in paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)
empty_paulis_opt = []
for ihd in range(nhd):
    circuit = QuantumCircuit(qr_circuit)
    empty_paulis_opt.append(circuit)
def ApplyEmptyPauliGate(ihd, qc_):
    for inst in empty_paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)

In [181]:
empty_paulis_opt[0].draw(idle_wires=False)

In [182]:
#circuit = QuantumCircuit(qr_circuit)
#ApplyControlledPauliGate(0,circuit)
#print(circuit)
#ApplyControlledPauliGate(1,circuit)
#print(circuit)

In [183]:
eigen_energies_ref = np.zeros((n_hamiltonians),dtype=float)

for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    eigen_energies_ref[i_inter] = - J/4 * Delta * (n_dimer)
    # inter-dimer energies
    eigen_energies_ref[i_inter] += - J/4 * Delta * t_inter* (n_dimer-1) # open boundary condition

In [184]:
print(eigen_energies_ref)

[0.5  0.75]


In [185]:
e_dimer = -0.25 * J * (2.0-Delta)
print (e_dimer)

-0.75


In [186]:
norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)


eigen_energies_qzmc[0]  = e_dimer*n_dimer
print(eigen_energies_qzmc[0])

-1.5


In [187]:
pauli_identity = 'I'*n_qubit

In [188]:
#def fake_run_estimator (max_circuits_, estimator_, pubs_):
#    len_circuits_ = len(pubs_)
#    num_jobs_     = len_circuits_//max_circuits_
#    remainder_    = len_circuits_%max_circuits_
#    if (remainder_>0):
#        num_jobs_ += 1
#    i_start_ = 0
#    list_out = []
#    for _ in range(num_jobs_):
#        i_end_ = min(i_start_ + max_circuits_,len_circuits_)
#        job_ = estimator_.run(pubs_[i_start_:i_end_])
#        job_result = job_.result()
#        len_result = len(job_result)
#        for i in range(len_result):
#            list_out.append(job_result[i].data.evs)
#        i_start_ += max_circuits_
#    return list_out

In [189]:
#def run_estimator (max_circuits_, estimator_, pubs_):
#    len_circuits_ = len(pubs_)
#    num_jobs_     = len_circuits_//max_circuits_
#    remainder_    = len_circuits_%max_circuits_
#    if (remainder_>0):
#        num_jobs_ += 1
#    i_start_ = 0
#    job_ids_ = []
#    for _ in range(num_jobs_):
#        i_end_ = min(i_start_ + max_circuits_,len_circuits_)
#        job_ = estimator_.run(pubs_[i_start_:i_end_])
#        job_ids_.append(job_.job_id())
#        i_start_ += max_circuits_
#    return job_ids_

In [190]:
# test
import copy
from scipy import sparse
from datetime import datetime
def generate_numbers_with_ones(n, k):
    if k > n:
        return []
    if k==n:
        return [(1<<n-1)]
    if k==0:
        return [0]
    # Start with the lowest number with exactly k ones (e.g., for n=5, k=2: "00011")
    number = (1 << k) - 1  # All 1s
    m = n-k

    results = []

    # The highest number with n-m ones and m zeros (e.g., for n=5, m=2: "11100")
    limit = (1 << n) - (1 << m)

    while number <= limit:
        # Count 0s to ensure it has exactly m zeros
        binary_str = f'{number:0{n}b}'
        # Check if the binary representation of the number has exactly m zeros
        if binary_str.count('0') == m:
            results.append(number)
        
        # Gosper's hack to get the next combination of (n-m) 1s in an n-bit number
        c = number & -number
        r = number + c
        number = (((r ^ number) >> 2) // c) | r

    return results

# sectorize
nsec = n_qubit//2+1
dim_sub = [0 for _ in range(nsec)]
indx_sub = [[] for _ in range(nsec)]
iindx_sub = [[] for _ in range(nsec)]

Z_target = [0 for _ in range(nsec)]

for isec in range(nsec):
    Z_target[isec] = (n_qubit-2*isec)

for isec in range(nsec):
    dim_sub[isec] = 0
    indx_sub[isec] = generate_numbers_with_ones(n_qubit,isec)
    dim_sub[isec] = len(indx_sub[isec])
    
    #print(indx_sub[isec])


    if (core==0):
        st = '# dimension of subspace with Z= {Z_target}: {dim_sub}'.format(Z_target=Z_target[isec],dim_sub=dim_sub[isec])
        print(st)
    indx_sub[isec] = np.array(indx_sub[isec])
    # inverse of indx_sub
    #print(indx_sub[isec])
    iindx_sub[isec] = -np.ones((dim),dtype=int)
    for i in range(dim_sub[isec]):
        iindx_sub[isec][indx_sub[isec][i]] = i
# exact eigenvalues
eigen_energies_exact   = []
eigen_vectors_exact   = []
for isec in range(nsec):
    eigen_energies_exact.append(np.zeros((n_inter,dim_sub[isec]),dtype=float))
    eigen_vectors_exact.append(np.zeros((n_inter,dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    for alpha in range(n_hamiltonians):
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        # diagonalize sectorized hamiltonian
        eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
        #if (isec==nsec-1):
        if (np.abs(Z_target[isec])<1e-10):
            print(alpha, eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_exact[isec][alpha,:]   = eigen_e
        eigen_vectors_exact[isec][alpha,:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        #if (core==0):
        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
        #    memory_usage(st)

def ExactGaussian (isec, alpha, eps, beta):
    Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=float)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-0.5 * beta ** 2 * (eigen_energies_exact[isec][alpha,k]-eps)**2)
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def ExactEvolution (isec, alpha, eps, time):
    Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_exact[isec][alpha,k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol
# intra-dimer terms
hamiltonian_list = []
for i in range(0,n_qubit,2):
    ii = i
    #jj = (i+1)%n_qubit # periodic boundary condition
    jj = (i+1) # open boundary condition
    if (jj>=n_qubit):
        continue
    term = ('XX',[ii,jj],-J/4)
    hamiltonian_list.append(term)
    term = ('YY',[ii,jj],-J/4)
    hamiltonian_list.append(term)
    term = ('ZZ',[ii,jj],-J*Delta/4)
    hamiltonian_list.append(term)
    term = ('Z',[ii],-h/2)
    hamiltonian_list.append(term)
    term = ('Z',[jj],-h/2)
    hamiltonian_list.append(term)
# inter-dimer terms
hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
hamiltonian_intra = hamiltonian.simplify()
if (core==0):
    print(single_line(str(hamiltonian_intra)))
    print('')

hamiltonians_inter       = []
for alpha in range(n_hamiltonians):
    hamiltonian_list = []
    hamiltonians_inter.append((hamiltonians[alpha]-hamiltonian_intra).simplify())

    if (core==0):
        print(alpha, single_line(str(hamiltonians_inter[alpha])))
        print('')
# exact eigenvalues for intra-dimer parts
eigen_energies_intra   = []
eigen_vectors_intra  = []
for isec in range(nsec):
    eigen_energies_intra.append(np.zeros((dim_sub[isec]),dtype=float))
    eigen_vectors_intra.append(np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    start_time = datetime.now()
    # project hamiltonian on to specified sector
    H_sparse = hamiltonian_intra.to_matrix(sparse=True)
    H_sparse.eliminate_zeros()
    jsec = isec
    row      = []
    col      = []
    data     = []
    for ii in range(dim_sub[isec]):
        i = indx_sub[isec][ii]
        for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
            # j is always in indx_sub[isec], because the Hamiltonian does not mix it
            #print(i,j)
            j = H_sparse.indices[ind]
            jj = iindx_sub[jsec][j]
            row.append(jj)
            col.append(ii)
            #print(ii,jj)
            data.append(H_sparse.data[ind])
    H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

    # diagonalize sectorized hamiltonian
    eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
    #if (isec==nsec-1):
    if (np.abs(Z_target[isec])<1e-10):
        print(eigen_e[0],eigen_e[1] )
    #    if (alpha==6):
    #        for k in range(dim_sub[isec]):
    #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
    #            print(k,np.abs(overlap)**2,eigen_e[k])
    #
    eigen_energies_intra[isec][:]   = eigen_e
    eigen_vectors_intra[isec][:,:] = eigen_v
    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    #if (core==0):
    #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
    #    memory_usage(st)
# exact eigenvalues for inter-dimer parts
eigen_energies_inter   = []
eigen_vectors_inter  = []
for isec in range(nsec):
    eigen_energies_inter.append(np.zeros((n_inter,dim_sub[isec]),dtype=float))
    eigen_vectors_inter.append(np.zeros((n_inter,dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    for alpha in range(n_hamiltonians):
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonians_inter[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        # diagonalize sectorized hamiltonian
        eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
        #if (isec==nsec-1):
        if (np.abs(Z_target[isec])<1e-10):
            print(alpha, eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_inter[isec][alpha,:]   = eigen_e
        eigen_vectors_inter[isec][alpha,:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        #if (core==0):
        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
        #    memory_usage(st)

def ExactEvolution_intra (isec, eps, time):
    Vl = copy.deepcopy(eigen_vectors_intra[isec][:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_intra[isec][k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def ExactEvolution_inter (isec, alpha, eps, time):
    Vl = copy.deepcopy(eigen_vectors_inter[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_inter[isec][alpha,k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def TrotterEvolution(isec, alpha, time, eps, n_trotter, indx):
    dt = time/n_trotter
    if (indx==0): 
        u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt/2.0)
        u_trotter = ExactEvolution_intra (isec, 0.0, dt)@u_trotter
        for i_trotter in range(n_trotter-1):
            u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt)@u_trotter
            u_trotter = ExactEvolution_intra (isec, 0.0, dt)@u_trotter
        u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt/2.0)@u_trotter
    elif (indx==1):
        u_trotter = ExactEvolution_intra (isec, 0.0, dt/2.0)
        u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt)@u_trotter
        for i_trotter in range(n_trotter-1):
            u_trotter = ExactEvolution_intra (isec, 0.0, dt)@u_trotter
            u_trotter = ExactEvolution_inter (isec, alpha, 0.0, dt)@u_trotter
        u_trotter = ExactEvolution_intra (isec, 0.0, dt/2.0)@u_trotter
    return u_trotter*np.exp(1j*eps*time)

n_trotter = 2

norms_approximate  = np.ones((nsec,n_hamiltonians),dtype=float)
#for isec in range(nsec):
for isec in range(nsec-1,nsec):
    # 
    H_subs = [0 for _ in range(n_hamiltonians)]
    for alpha in range(n_hamiltonians):
        H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))
        H_subs[alpha] = H_sub.toarray()


    phi = eigen_vectors_exact[isec][0,:,0]
    E_approximate = eigen_energies_exact[isec][0,0]
    eps = np.real(phi.conj()@H_subs[1]@phi)
    eigen_energies_test = np.zeros((n_hamiltonians),dtype=float)
    for alpha in range(1,n_hamiltonians):
        # norm calculation (i_obs = 0)
        i_obs = 0
        norm = 0.0
        for imc in range(nmc):
            times = O_timelists[alpha][i_obs][imc]
            #indx = 0
            #indx_dag = 0 # same for second order trotter
            #evol_norm = np.identity(dim_sub[isec])
            #i_time = 0
            #for alpha_ in range(1,alpha):
            #    evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx)@evol_norm
            #    i_time += 1
            #evol_norm = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_norm
            #i_time += 1
            ##evol_norm = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_norm
            ##i_time += 1
            #for alpha_ in reversed(range(1,alpha)):
            #    evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx_dag)@evol_norm
            #    i_time += 1
            #norm += phi.conj()@evol_norm@phi

            indx = 1
            indx_dag = 1
            evol_norm = np.identity(dim_sub[isec])
            i_time = 0
            for alpha_ in range(1,alpha):
                evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx)@evol_norm
                i_time += 1
            evol_norm = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_norm
            i_time += 1
            #evol_norm = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_norm
            #i_time += 1
            for alpha_ in reversed(range(1,alpha)):
                evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx_dag)@evol_norm
                i_time += 1
            norm += phi.conj()@evol_norm@phi

        norm /= (nmc)
        norm = norm.real
        print(alpha,norm)

        # dE1 calculation (i_obs = 1)
        i_obs = 1
        dE = 0.0
        for imc in range(nmc):
            times = O_timelists[alpha][i_obs][imc]
            #indx = 0
            #indx_dag = 0
            #evol_dE = np.identity(dim_sub[isec])
            #i_time = 0
            #for alpha_ in range(1,alpha):
            #    evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx)@evol_dE
            #    i_time += 1
            #evol_dE = (H_subs[alpha]-H_subs[alpha-1])@evol_dE
            #evol_dE = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_dE
            #i_time += 1
            ##evol_dE = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_dE
            ##i_time += 1
            #for alpha_ in reversed(range(1,alpha)):
            #    evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_exact[isec][alpha_,0],n_trotter,indx_dag)@evol_dE
            #    i_time += 1
            #dE += phi.conj()@evol_dE@phi

            indx = 1
            indx_dag = 1
            evol_dE = np.identity(dim_sub[isec])
            i_time = 0
            for alpha_ in range(1,alpha):
                evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx)@evol_dE
                i_time += 1
            evol_dE = (H_subs[alpha]-H_subs[alpha-1])@evol_dE
            evol_dE = TrotterEvolution(isec, alpha, times[i_time], eps ,n_trotter,indx)@evol_dE
            i_time += 1
            #evol_dE = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_exact[isec][alpha,0],n_trotter,indx_dag)@evol_dE
            #i_time += 1
            for alpha_ in reversed(range(1,alpha)):
                evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx_dag)@evol_dE
                i_time += 1
            dE += phi.conj()@evol_dE@phi

        dE /= (nmc)
        dE /= norm
        dE = dE.real
        E_approximate += dE
        print(alpha,E_approximate-eigen_energies_exact[isec][alpha,0])
        print(E_approximate, eigen_energies_exact[isec][0,0], eps)
        eigen_energies_test[alpha] = E_approximate

        if (alpha<n_hamiltonians-1):
            i_obs = 0
            norm2 = 0.0
            for imc in range(nmc):
                times = O_timelists[alpha][i_obs][imc]

                indx = 1
                indx_dag = 1
                evol_norm = np.identity(dim_sub[isec])
                i_time = 0
                for alpha_ in range(1,alpha):
                    evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx)@evol_norm
                    i_time += 1
                evol_norm = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_test[alpha], n_trotter,indx)@evol_norm
                i_time += 1
                evol_norm = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_test[alpha], n_trotter,indx_dag)@evol_norm
                i_time += 1
                for alpha_ in reversed(range(1,alpha)):
                    evol_norm = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_], n_trotter,indx_dag)@evol_norm
                    i_time += 1
                norm2 += phi.conj()@evol_norm@phi

            norm2 /= (nmc)
            norm2 = norm2.real
            print(alpha,norm2)


            # dE2 calculation (i_obs = 2)
            i_obs = 2
            dE2 = 0.0
            for imc in range(nmc):
                times = O_timelists[alpha][i_obs][imc]

                indx = 1
                indx_dag = 1
                evol_dE = np.identity(dim_sub[isec])
                i_time = 0
                for alpha_ in range(1,alpha):
                    evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx)@evol_dE
                    i_time += 1
                evol_dE = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_test[alpha],n_trotter,indx)@evol_dE
                i_time += 1
                evol_dE = (H_subs[alpha+1]-H_subs[alpha])@evol_dE
                evol_dE = TrotterEvolution(isec, alpha, times[i_time], eigen_energies_test[alpha],n_trotter,indx_dag)@evol_dE
                i_time += 1
                for alpha_ in reversed(range(1,alpha)):
                    evol_dE = TrotterEvolution(isec, alpha_, times[i_time], eigen_energies_test[alpha_],n_trotter,indx_dag)@evol_dE
                    i_time += 1
                dE2 += phi.conj()@evol_dE@phi

            dE2 /= (nmc)
            dE2 /= norm2
            dE2 = dE2.real
            eps = E_approximate + dE2
            print(alpha,eps-eigen_energies_exact[isec][alpha+1,0])

        #if (core==0):
        #    print(alpha, norms_approximate[isec,alpha])

# read exact values
nsec = n_qubit//2 + 1
#exact_dir = '../exact'
#exact_dir = '../../4/exact'
exact_dir = '../../'+str(n_site)+'/exact'
norms_exact = np.zeros((nsec,n_hamiltonians), dtype=float)
eigen_energies_exact = np.zeros((nsec,n_hamiltonians),dtype=float)
with open(exact_dir + '/exact.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        for isec in range(nsec):
            norms_exact[isec,alpha] = float(ls[isec+1])
            eigen_energies_exact[isec,alpha] = float(ls[isec+nsec+1])
        alpha += 1
#for isec in range(nsec):
#    for alpha in range(n_hamiltonians):
#        print(isec,alpha,norms_exact[isec,alpha],eigen_energies_exact[isec,alpha])



# dimension of subspace with Z= 4: 1
# dimension of subspace with Z= 2: 4
# dimension of subspace with Z= 0: 6
0 -1.5 -0.5000000000000001
1 -1.6160254037844386 -0.9571067811865472
SparsePauliOp(['IIXX', 'IIYY', 'IIZZ', 'XXII', 'YYII', 'ZZII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])

0 SparsePauliOp(['IIII'],              coeffs=[0.+0.j])

1 SparsePauliOp(['IXXI', 'IYYI', 'IZZI'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j])

-1.5 -0.5000000000000001
0 0.0 0.0
1 -0.75 -0.75
1 0.8976076561253109
1 -0.025870429410131957
-1.6418958331945706 -1.5 -1.5000000000000004


In [191]:
#from qiskit.quantum_info import Operator
#unitary = Operator(init_circuit).data
#vec_0 = np.zeros((dim*2),dtype=complex)
#vec_0[0] = 1
#vec_0 = unitary@vec_0
#
#vec_1 = np.zeros((dim*2),dtype=complex)
#vec_1[1] = 1
#vec_1 = unitary@vec_1
##
#vecc = vec_0[0::2]
#for k in range(dim):
#    if (np.abs(vecc[k])>1e-10):
#        print(k,vecc[k])
#vecc = vec_1[1::2]
#for k in range(dim):
#    if (np.abs(vecc[k])>1e-10):
#        print(k,vecc[k])
#print(vec_0[0::2].conj()@hamiltonians[0].to_matrix()@vec_0[0::2])
#print(vec_1[1::2].conj()@hamiltonians[0].to_matrix()@vec_1[1::2])
#
#
## checking validity of init matrix -> checked

In [192]:
#from qiskit.providers import JobStatus
#import time as time_lib
#
#def moniter_jobs (service_, job_ids_):
#    num_jobs_       = len(job_ids_)
#    done_           = False
#    problem_        = False
#    while not done_:
#        done_ = True
#        i_start_ = 0
#        for ind_job_ in range(num_jobs_):
#            job_id_ = job_ids_[ind_job_]
#            job_ = service_.job(job_id_)
#            if job_.status() is JobStatus.DONE:
#                continue
#            else:
#                done_ = False
#                # exit if there is a problem
#                if job_.status() in [JobStatus.ERROR, JobStatus.CANCELLED]:
#                    s_ = ''
#                    s_ += '## There was a problem in submitting job_ids[{ind_job}], '.format(ind_job=ind_job_)
#                    s_ +='\n'
#                    s_ += '  {}'.format(job_id_)
#                    s_ +='\n'
#                    print(s_)
#                    problem_ = True
#                if (problem_):
#                    break
#        if (problem_):
#            break
#        time_lib.sleep(30)
#    return [done_, problem_]
#
#def accumulate_job_results (service_, job_ids_):
#    list_out = []
#    for job_id_ in job_ids_:
#        job_ = service_.job(job_id_)
#        while job_.status() is not JobStatus.DONE:
#            time_lib.sleep(30) 
#        job_result = job_.result()
#        len_result = len(job_result)
#        for i in range(len_result):
#            list_out.append(job_result[i].data.evs)
#    return list_out
#

In [193]:
run_circuits_file = 'circuits_opt.qpy'
if (core==0):
    code_run_estimator = '''from qiskit_aer.primitives import EstimatorV2 as Estimator
from qiskit import qpy
import sys, pickle
from qiskit.quantum_info import SparsePauliOp


pickled_input_data = sys.stdin.buffer.read()
input_data = pickle.loads(pickled_input_data)
n_qubit ='''+ str(n_qubit) + '''
observable = SparsePauliOp.from_sparse_list([("Z", ['''+str(indx_qm)+'''], 1)], num_qubits='''+str(n_qubit_circuit)+''')
backend_options = '''+ str(backend_options) +'''
estimator = Estimator(options = {"default_precision":0.00,
                                     'backend_options':backend_options})
run_circuits_file = \'''' + run_circuits_file + '''\'
with open(run_circuits_file, 'rb') as file_:
    circuits = qpy.load(file_)
pubs = []
len_input = len(input_data)
for ind in range(len_input):
    i0     = input_data[ind][0]
    if (len(input_data[ind])>1):
        params = input_data[ind][1]
        pubs.append((circuits[i0],observable,params))
    else:
        pubs.append((circuits[i0],observable))
job = estimator.run(pubs)
result = job.result()
len_result = len(result)
list_out = []
for i in range(len_result):
    list_out.append(result[i].data.evs)
sys.stdout.buffer.write(pickle.dumps(list_out))
    '''
    
    with open('run_estimator.py', 'w') as file_:
        file_.write(code_run_estimator)

In [194]:
from datetime import datetime
from qiskit.circuit import ParameterVector

In [195]:
#job_ids_save = [None for _ in range(n_hamiltonians)]
#num_jobs_save = [None for _ in range(n_hamiltonians)]

In [196]:
# first run, dE1
alpha = 1
eps = eigen_energies_qzmc[0]
# first-order perturbation correction = 0

start_time = datetime.now()
circuits = []
times = ParameterVector('t', 2*alpha-1)

# implement quantum circuits by parts
circuit_segment = [None for _ in range(2)] 
circuit = QuantumCircuit(qr_circuit)
circuit.h(qm)
for inst in init_circuit.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)
i_time  = 0
phase   = 0.0
# \mathcal{P}_{\alpha-1}
evolution_times = []
alphas = []
n_trotters = []
for alpha_ in range(1,alpha):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
if (len(evolution_times)>0):
    phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[0]/n_trotters[0]/2
ApplyConsecutiveTrotterEvolution_from_dimer_solution(evolution_times,alphas,n_trotters,circuit)
circuit_segment[0] = circuit
#circuit_segment[0] = pm3.run(circuit)

circuit = QuantumCircuit(qr_circuit)

evolution_times = []
alphas = []
n_trotters = []

# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
#print(evolution_times,alphas,n_trotters)
phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

# \mathcal{P}^{\dagger}_{\alpha-1}
for alpha_ in reversed(range(1,alpha)):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
ApplyConsecutiveTrotterEvolution_to_dimer_solution(evolution_times,alphas,n_trotters,circuit)
phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[-1]/n_trotters[-1]/2
#print(circuit)

for inst in init_circuit_inv.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)

circuit.rz(phase,qm)
circuit.h(qm)

#circuit_segment[indx][1] = pm3.run(circuit)
circuit_segment[1] = circuit

nhd = len(hamiltonian_diffs_reduced_list[alpha-1])

for ihd in range(nhd):
    # norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyEmptyPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt

for ihd in range(nhd):
    # dE*norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyControlledPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt


# 
# pubs = []
# 
# n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
# n_pubs_for_ = [0 for _ in range(cores)]
# remainder         = n_pubs%cores
# for i_core in range(cores):
#     n_pubs_for_[i_core] = n_pubs//cores
#     if (i_core<remainder):
#         n_pubs_for_[i_core] += 1
# if (core==0 and alpha==1):
#     print('# of different quantum circuits to run = ', n_pubs)
# 
# i_start         = sum(n_pubs_for_[:core])
# i_end           = i_start + n_pubs_for_[core]
# ind_pub         = 0
# indx_circuit    = 0
# ## dE1
# for ihd in range(nhd):
#     # norm, indx = 0, 1
#     i_obs = 0
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1
#     # dE*norm, indx = 0, 1
#     i_obs = 1
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1

if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    del circuits[:]
    subprocess.run(['python3', 'run_pass_manager.py'])
#comm.Barrier()

estimator_inputs = []

n_pubs = 0
# dE1
n_pubs += (2*nhd) * nmc  # nhd for norm, nhd for dE

n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1
if (core==0 and alpha==1):
    print('# of different quantum circuits to run = ', n_pubs)

i_start         = sum(n_pubs_for_[:core])
i_end           = i_start + n_pubs_for_[core]
ind_pub         = 0
indx_circuit    = 0
## norm
i_obs = 0
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
        ind_pub += 1
    indx_circuit += 1
## dE*norm
i_obs = 1
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
        ind_pub += 1
    indx_circuit += 1


result_values_core = [0.0 for _ in range(n_pubs_for_[core])]


#job_ids = run_estimator(max_circuits, estimator, pubs)
#result = fake_run_estimator(max_circuits, estimator, pubs)
#job_ids_save[alpha] = job_ids
#num_jobs_save[alpha] = len(job_ids)
#with open('XXZ4.job_ids','w') as file_:
#    for alpha_ in range(1, n_hamiltonians):
#        s = ''
#        for job_id in job_ids_save[alpha_]:
#            s += '  {}'.format(job_id)
#        s += '\n'
#        file_.write(s)
#[done, problem] = moniter_jobs (service, job_ids_save[alpha])
#result = accumulate_job_results(service, job_ids_save[alpha])

#gc.collect()
#
#for i in range(n_pubs_for_[core]):
#    result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
#del result

gc.collect()
serialized_estimator_inputs = pickle.dumps(estimator_inputs)
estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
               stdout=subprocess.PIPE,stderr=subprocess.PIPE)



# of different quantum circuits to run =  600


In [197]:
try:
    result = pickle.loads(estimator_run.stdout)
except EOFError: # empty list
    result = []
gc.collect()

for i in range(n_pubs_for_[core]):
    #print(result[i])
    computed_value = result[i]

    # # shot errors
    #p_up = (computed_value + 1.0)/2.0
    #sample = np.random.binomial(n_shot,p_up)
    #shot_error = 2*(sample/(n_shot) - p_up)
    #computed_value += shot_error

    result_values_core[i] = computed_value
#print(core,result_values_core)
del result


### bcast
#comm.Barrier()
result_values = []
for i_core in range(cores):
    if (n_pubs_for_[i_core]==0):
        continue
    result_values_temp = result_values_core
    result_values += result_values_temp
    #comm.Barrier()
#print(result_values)

norms_1    = np.zeros((nhd),dtype=float)
pauli_norms_1  = np.zeros((nhd),dtype=float)
i_meas = 0
# norms
for ihd in range(nhd):
    for imc in range(nmc):
        norms_1[ihd] += result_values[i_meas]
        i_meas += 1
# paulis
for ihd in range(nhd):
    for imc in range(nmc):
        pauli_norms_1[ihd] += result_values[i_meas]
        i_meas += 1
norms_1 /= nmc
pauli_norms_1 /= nmc

print(norms_1)
print(pauli_norms_1)

# dE1
dE1 = 0.0
for ihd in range(nhd):
    pauli_exp = pauli_norms_1[ihd]/norms_1[ihd]
    coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
    dE1 += coeff * pauli_exp
dE1 = dE1.real

eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
norms_qzmc[alpha] = np.sum(norms_1)/(nhd)
del times
del result_values[:]
del result_values_core[:]
gc.collect()

end_time = datetime.now()
elapsed = end_time-start_time
elapsed = elapsed.total_seconds()
if (core==0):
    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
    memory_usage(st)
    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
    print(st)

[0.89760766]
[0.16982238]
[# 100.0%, elapsed time = 3.029731 secs] memory usage:    0.24868 GiB
1 0.8976076561253096 -0.025870429410131734
# 100.0%


In [101]:
# first run, dE2
alpha = 1

start_time = datetime.now()
circuits = []
times = ParameterVector('t', 2*alpha)

# implement quantum circuits by parts
circuit_segment = [None for _ in range(2)] 
circuit = QuantumCircuit(qr_circuit)
circuit.h(qm)
for inst in init_circuit.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)
i_time  = 0
phase   = 0.0
# \mathcal{P}_{\alpha-1}
evolution_times = []
alphas = []
n_trotters = []
for alpha_ in range(1,alpha):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
phase += (eigen_energies_qzmc[alpha]-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

if (len(evolution_times)>0):
    phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[0]/n_trotters[0]/2
ApplyConsecutiveTrotterEvolution_from_dimer_solution(evolution_times,alphas,n_trotters,circuit)
circuit_segment[0] = circuit
#circuit_segment[0] = pm3.run(circuit)

circuit = QuantumCircuit(qr_circuit)

evolution_times = []
alphas = []
n_trotters = []

# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
phase += (eigen_energies_qzmc[alpha]-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

# \mathcal{P}^{\dagger}_{\alpha-1}
for alpha_ in reversed(range(1,alpha)):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
ApplyConsecutiveTrotterEvolution_to_dimer_solution(evolution_times,alphas,n_trotters,circuit)
phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[-1]/n_trotters[-1]/2
#print(circuit)

for inst in init_circuit_inv.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)

circuit.rz(phase,qm)
circuit.h(qm)

#circuit_segment[indx][1] = pm3.run(circuit)
circuit_segment[1] = circuit

nhd = len(hamiltonian_diffs_reduced_list[alpha])

for ihd in range(nhd):
    # norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyEmptyPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer

for ihd in range(nhd):
    # dE*norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyControlledPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt


# 
# pubs = []
# 
# n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
# n_pubs_for_ = [0 for _ in range(cores)]
# remainder         = n_pubs%cores
# for i_core in range(cores):
#     n_pubs_for_[i_core] = n_pubs//cores
#     if (i_core<remainder):
#         n_pubs_for_[i_core] += 1
# if (core==0 and alpha==1):
#     print('# of different quantum circuits to run = ', n_pubs)
# 
# i_start         = sum(n_pubs_for_[:core])
# i_end           = i_start + n_pubs_for_[core]
# ind_pub         = 0
# indx_circuit    = 0
# ## dE1
# for ihd in range(nhd):
#     # norm, indx = 0, 1
#     i_obs = 0
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1
#     # dE*norm, indx = 0, 1
#     i_obs = 1
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1

if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    del circuits[:]
    subprocess.run(['python3', 'run_pass_manager.py'])
#comm.Barrier()

estimator_inputs = []

n_pubs = 0
# dE1
n_pubs += (2*nhd) * nmc  # nhd for norm, nhd for dE

n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1
if (core==0 and alpha==1):
    print('# of different quantum circuits to run = ', n_pubs)

i_start         = sum(n_pubs_for_[:core])
i_end           = i_start + n_pubs_for_[core]
ind_pub         = 0
indx_circuit    = 0
## norm
i_obs = 0
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc])]
        ind_pub += 1
    indx_circuit += 1
## dE2*norm
i_obs = 2
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc])]
        ind_pub += 1
    indx_circuit += 1


result_values_core = [0.0 for _ in range(n_pubs_for_[core])]


#job_ids = run_estimator(max_circuits, estimator, pubs)
#result = fake_run_estimator(max_circuits, estimator, pubs)
#job_ids_save[alpha] = job_ids
#num_jobs_save[alpha] = len(job_ids)
#with open('XXZ4.job_ids','w') as file_:
#    for alpha_ in range(1, n_hamiltonians):
#        s = ''
#        for job_id in job_ids_save[alpha_]:
#            s += '  {}'.format(job_id)
#        s += '\n'
#        file_.write(s)
#[done, problem] = moniter_jobs (service, job_ids_save[alpha])
#result = accumulate_job_results(service, job_ids_save[alpha])

#gc.collect()
#
#for i in range(n_pubs_for_[core]):
#    result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
#del result

gc.collect()
serialized_estimator_inputs = pickle.dumps(estimator_inputs)
estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
               stdout=subprocess.PIPE,stderr=subprocess.PIPE)



IndexError: list index out of range

In [ ]:
try:
    result = pickle.loads(estimator_run.stdout)
except EOFError: # empty list
    result = []
gc.collect()

for i in range(n_pubs_for_[core]):
    #print(result[i])
    computed_value = result[i]

    # # shot errors
    #p_up = (computed_value + 1.0)/2.0
    #sample = np.random.binomial(n_shot,p_up)
    #shot_error = 2*(sample/(n_shot) - p_up)
    #computed_value += shot_error

    result_values_core[i] = computed_value
#print(core,result_values_core)
del result


### bcast
#comm.Barrier()
result_values = []
for i_core in range(cores):
    if (n_pubs_for_[i_core]==0):
        continue
    result_values_temp = result_values_core
    result_values += result_values_temp
    #comm.Barrier()
#print(result_values)

norms_2    = np.zeros((nhd),dtype=float)
pauli_norms_2  = np.zeros((nhd),dtype=float)
i_meas = 0
# norms
for ihd in range(nhd):
    for imc in range(nmc):
        norms_2[ihd] += result_values[i_meas]
        i_meas += 1
# paulis
for ihd in range(nhd):
    for imc in range(nmc):
        pauli_norms_2[ihd] += result_values[i_meas]
        i_meas += 1
norms_2 /= nmc
pauli_norms_2 /= nmc

print(norms_2)
print(pauli_norms_2)

# dE2
dE2 = 0.0
for ihd in range(nhd):
    pauli_exp = pauli_norms_2[ihd]/norms_2[ihd]
    coeff = hamiltonian_diffs_reduced_list[alpha][ihd][1]
    dE2 += coeff * pauli_exp
dE2 = dE2.real

eps = eigen_energies_qzmc[alpha] + dE2
del times
del result_values[:]
del result_values_core[:]
gc.collect()

end_time = datetime.now()
elapsed = end_time-start_time
elapsed = elapsed.total_seconds()
if (core==0):
    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
    memory_usage(st)
    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
    if (alpha<n_hamiltonians-1):
        print('precision of the predictor for next', eps-eigen_energies_exact[-1,alpha+1])
    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
    print(st)

[0.58590867 0.58590867]
[0.1695538  0.16424088]
[# 50.0%, elapsed time = 14.327486 secs] memory usage:    0.25058 GiB
1 0.7458153963699342 -0.03689108840046629
precision of the predictor for next -0.003124601255234616
# 50.0%


In [51]:
# second run, dE1
alpha = 2

start_time = datetime.now()
circuits = []
times = ParameterVector('t', 2*alpha-1)

# implement quantum circuits by parts
circuit_segment = [None for _ in range(2)] 
circuit = QuantumCircuit(qr_circuit)
circuit.h(qm)
for inst in init_circuit.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)
i_time  = 0
phase   = 0.0
# \mathcal{P}_{\alpha-1}
evolution_times = []
alphas = []
n_trotters = []
for alpha_ in range(1,alpha):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
if (len(evolution_times)>0):
    phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[0]/n_trotters[0]/2
ApplyConsecutiveTrotterEvolution_from_dimer_solution(evolution_times,alphas,n_trotters,circuit)
circuit_segment[0] = circuit
#circuit_segment[0] = pm3.run(circuit)

circuit = QuantumCircuit(qr_circuit)

evolution_times = []
alphas = []
n_trotters = []

# P_{\alpha}
alphas.append(alpha)
evolution_times.append(times[i_time])
n_trotters.append(NumberOfTrotterSteps(alpha))
#print(evolution_times,alphas,n_trotters)
phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
i_time += 1

# \mathcal{P}^{\dagger}_{\alpha-1}
for alpha_ in reversed(range(1,alpha)):
    alphas.append(alpha_)
    evolution_times.append(times[i_time])
    n_trotters.append(NumberOfTrotterSteps(alpha_))
    phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
    i_time += 1
#ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,1,circuit)
ApplyConsecutiveTrotterEvolution_to_dimer_solution(evolution_times,alphas,n_trotters,circuit)
phase -= (eigen_energies_qzmc[0]-eigen_energies_ref[0]) * evolution_times[-1]/n_trotters[-1]/2
#print(circuit)

for inst in init_circuit_inv.data:
    circuit.append(inst.operation, inst.qubits, inst.clbits)

circuit.rz(phase,qm)
circuit.h(qm)

#circuit_segment[indx][1] = pm3.run(circuit)
circuit_segment[1] = circuit

nhd = len(hamiltonian_diffs_reduced_list[alpha-1])

for ihd in range(nhd):
    # norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyEmptyPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt

for ihd in range(nhd):
    # dE*norm for ihd
    circuit = QuantumCircuit(qr_circuit)
    #circuit = pm3.run(circuit)
    for inst in circuit_segment[0].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    ApplyControlledPauliGate(ihd,circuit)
    for inst in circuit_segment[1].data:
        circuit.append(inst.operation, inst.qubits, inst.clbits)
    circuits.append(circuit) # aer
    #circuit_opt = pm1.run(circuit)
    #circuits.append(circuit_opt)
    #del circuit_opt


# 
# pubs = []
# 
# n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
# n_pubs_for_ = [0 for _ in range(cores)]
# remainder         = n_pubs%cores
# for i_core in range(cores):
#     n_pubs_for_[i_core] = n_pubs//cores
#     if (i_core<remainder):
#         n_pubs_for_[i_core] += 1
# if (core==0 and alpha==1):
#     print('# of different quantum circuits to run = ', n_pubs)
# 
# i_start         = sum(n_pubs_for_[:core])
# i_end           = i_start + n_pubs_for_[core]
# ind_pub         = 0
# indx_circuit    = 0
# ## dE1
# for ihd in range(nhd):
#     # norm, indx = 0, 1
#     i_obs = 0
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1
#     # dE*norm, indx = 0, 1
#     i_obs = 1
#     for indx in range(2):
#         for imc in range(nmc):
#             my_turn = ind_pub>=i_start and ind_pub<i_end
#             if (my_turn):
#                 pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
#             ind_pub += 1
#         indx_circuit += 1

if (core==0):
    with open ('circuits.qpy', 'wb') as file_:
        qpy.dump(circuits,file_)
    del circuits[:]
    subprocess.run(['python3', 'run_pass_manager.py'])
#comm.Barrier()

estimator_inputs = []

n_pubs = 0
# dE1
n_pubs += (2*nhd) * nmc  # nhd for norm, nhd for dE

n_pubs_for_ = [0 for _ in range(cores)]
remainder         = n_pubs%cores
for i_core in range(cores):
    n_pubs_for_[i_core] = n_pubs//cores
    if (i_core<remainder):
        n_pubs_for_[i_core] += 1
if (core==0 and alpha==1):
    print('# of different quantum circuits to run = ', n_pubs)

i_start         = sum(n_pubs_for_[:core])
i_end           = i_start + n_pubs_for_[core]
ind_pub         = 0
indx_circuit    = 0
## norm
i_obs = 0
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
        ind_pub += 1
    indx_circuit += 1
## dE*norm
i_obs = 1
for ihd in range(nhd):
    for imc in range(nmc):
        my_turn = ind_pub>=i_start and ind_pub<i_end
        if (my_turn):
            estimator_inputs += [(indx_circuit,O_timelists[alpha][i_obs][imc][:-1])]
        ind_pub += 1
    indx_circuit += 1


result_values_core = [0.0 for _ in range(n_pubs_for_[core])]


#job_ids = run_estimator(max_circuits, estimator, pubs)
#result = fake_run_estimator(max_circuits, estimator, pubs)
#job_ids_save[alpha] = job_ids
#num_jobs_save[alpha] = len(job_ids)
#with open('XXZ4.job_ids','w') as file_:
#    for alpha_ in range(1, n_hamiltonians):
#        s = ''
#        for job_id in job_ids_save[alpha_]:
#            s += '  {}'.format(job_id)
#        s += '\n'
#        file_.write(s)
#[done, problem] = moniter_jobs (service, job_ids_save[alpha])
#result = accumulate_job_results(service, job_ids_save[alpha])

#gc.collect()
#
#for i in range(n_pubs_for_[core]):
#    result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
#del result

gc.collect()
serialized_estimator_inputs = pickle.dumps(estimator_inputs)
estimator_run=subprocess.run(['python3', 'run_estimator.py'],input=serialized_estimator_inputs,
               stdout=subprocess.PIPE,stderr=subprocess.PIPE)



In [52]:
try:
    result = pickle.loads(estimator_run.stdout)
except EOFError: # empty list
    result = []
gc.collect()

for i in range(n_pubs_for_[core]):
    #print(result[i])
    computed_value = result[i]

    # # shot errors
    #p_up = (computed_value + 1.0)/2.0
    #sample = np.random.binomial(n_shot,p_up)
    #shot_error = 2*(sample/(n_shot) - p_up)
    #computed_value += shot_error

    result_values_core[i] = computed_value
#print(core,result_values_core)
del result


### bcast
#comm.Barrier()
result_values = []
for i_core in range(cores):
    if (n_pubs_for_[i_core]==0):
        continue
    result_values_temp = result_values_core
    result_values += result_values_temp
    #comm.Barrier()
#print(result_values)

norms_1    = np.zeros((nhd),dtype=float)
pauli_norms_1  = np.zeros((nhd),dtype=float)
i_meas = 0
# norms
for ihd in range(nhd):
    for imc in range(nmc):
        norms_1[ihd] += result_values[i_meas]
        i_meas += 1
# paulis
for ihd in range(nhd):
    for imc in range(nmc):
        pauli_norms_1[ihd] += result_values[i_meas]
        i_meas += 1
norms_1 /= nmc
pauli_norms_1 /= nmc

print(norms_1)
print(pauli_norms_1)

# dE1
dE1 = 0.0
for ihd in range(nhd):
    pauli_exp = pauli_norms_1[ihd]/norms_1[ihd]
    coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
    dE1 += coeff * pauli_exp
dE1 = dE1.real

eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
norms_qzmc[alpha] = np.sum(norms_1)/(nhd)
del times
del result_values[:]
del result_values_core[:]
gc.collect()

end_time = datetime.now()
elapsed = end_time-start_time
elapsed = elapsed.total_seconds()
if (core==0):
    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
    memory_usage(st)
    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
    print(st)

[0.50190114 0.50190114]
[0.15115243 0.14173032]
[# 100.0%, elapsed time = 18.064495 secs] memory usage:    0.25058 GiB
2 0.5019011365208734 -0.008315508755019074
# 100.0%
